In [4]:
!pip install google-genai

In [5]:
import os
import google.generativeai as genai
from google.colab import userdata

api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = api_key

genai.configure(api_key=api_key)
MODEL_NAME = "gemini-2.5-flash"

In [6]:
MODEL_CONFIG = {
    "technical": "You are a rigorous, elite software engineer. Focus heavily on code accuracy, precise debugging steps, and technical correctness. Keep prose minimal.",
    "billing": "You are an empathetic, professional billing support agent. Address financial concerns gently, explain refund/charge policies clearly, and prioritize customer satisfaction.",
    "general": "You are a helpful, friendly customer support assistant. Answer casual inquiries conversationally."
}

def get_crypto_price(asset: str) -> str:
    prices = {"bitcoin": "$65,432.10", "ethereum": "$3,450.00"}
    asset = asset.lower()
    if "bitcoin" in asset:
        return f"[System Tool Output]: The current price of Bitcoin is {prices['bitcoin']}."
    elif "ethereum" in asset:
        return f"[System Tool Output]: The current price of Ethereum is {prices['ethereum']}."
    return "[System Tool Output]: Price data not available for that asset."

In [7]:
def route_prompt(user_input: str) -> str:
    routing_system_prompt = (
        "You are an intent classification engine. Classify the user's text into ONE of these "
        "exact categories: 'technical', 'billing', 'tool', or 'general'. \n"
        "- Use 'technical' for code, bugs, or engineering issues.\n"
        "- Use 'billing' for refunds, charges, or account money issues.\n"
        "- Use 'tool' if the user asks for the current price of Bitcoin or Crypto.\n"
        "- Use 'general' for everything else.\n"
        "Return ONLY the category name in lowercase without punctuation. No extra text."
    )

    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=routing_system_prompt
    )

    response = model.generate_content(
        user_input,
        generation_config=genai.types.GenerationConfig(
            temperature=0.0,
            max_output_tokens=10,
        )
    )

    category = response.text.strip().lower()

    valid_categories = ["technical", "billing", "tool", "general"]
    if category not in valid_categories:
        category = "general"

    return category

In [8]:
def process_request(user_input: str) -> dict:
    category = route_prompt(user_input)

    if category == "tool":
        final_answer = get_crypto_price(user_input)
    else:
        system_prompt = MODEL_CONFIG[category]

        expert_model = genai.GenerativeModel(
            model_name=MODEL_NAME,
            system_instruction=system_prompt
        )

        response = expert_model.generate_content(
            user_input,
            generation_config=genai.types.GenerationConfig(
                temperature=0.7,
            )
        )
        final_answer = response.text

    return {
        "user_query": user_input,
        "routed_expert": category,
        "response": final_answer
    }

In [10]:
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month. I need a refund immediately.",
    "Hey! What are your business hours?",
    "What is the current price of Bitcoin?"
]

for query in test_queries:
    print(f"\nUser: {query}")
    try:
        result = process_request(query)
        print(f"--> Routed to: [{result['routed_expert'].upper()}] Expert")
        print(f"--> Response: {result['response']}\n")
    except ValueError as e:
        print(f"--> Error processing request for '{query}': {e}")
        print("--> This query might have been blocked by safety filters. Defaulting to 'general' category.\n")
    print("-" * 50)


User: My python script is throwing an IndexError on line 5.
--> Error processing request for 'My python script is throwing an IndexError on line 5.': Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 2.
--> This query might have been blocked by safety filters. Defaulting to 'general' category.

--------------------------------------------------

User: I was charged twice for my subscription this month. I need a refund immediately.
--> Error processing request for 'I was charged twice for my subscription this month. I need a refund immediately.': Invalid operation: The `response.text` quick accessor requires the response to contain a valid `Part`, but none were returned. The candidate's [finish_reason](https://ai.google.dev/api/generate-content#finishreason) is 2.
--> This query might have been blocked by safety filters